In [ ]:
from pathlib import Path
from datetime import datetime
import os, json, subprocess

# ========= CONFIG =========
SUBSCRIPTION_ID = "00000000-0000-0000-0000-000000000000"
RESOURCE_GROUP = "sample-resource-group"
LOCATION = "eastus"
SKU = "S"   # or "F0" if available in your subscription/region

LANGUAGE_RESOURCE_NAME = "sample-resource-name"

AZ_EXE = r"C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin\az.cmd"

# ========= AZ HELPERS =========
if not os.path.exists(AZ_EXE):
    raise FileNotFoundError(f"Azure CLI executable not found at: {AZ_EXE}")

def az(args, expect_json=True):
    cmd = [AZ_EXE] + args + ["-o", "json" if expect_json else "tsv"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"Azure CLI command failed:\n"
            f"CMD: {' '.join(cmd)}\n"
            f"STDOUT:\n{result.stdout}\n"
            f"STDERR:\n{result.stderr}"
        )
    return json.loads(result.stdout) if expect_json else result.stdout.strip()

def az_try(args, expect_json=True):
    cmd = [AZ_EXE] + args + ["-o", "json" if expect_json else "tsv"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        return None
    return json.loads(result.stdout) if expect_json else result.stdout.strip()

print("Checking Azure CLI login...")
acct = az_try(["account", "show"])
if acct is None:
    raise RuntimeError(
        "Azure CLI is reachable, but not logged in.\n"
        "Run in terminal:\n"
        "  az login\n"
        "or:\n"
        "  az login --use-device-code"
    )

print("Logged in as:", acct.get("user", {}).get("name"))
print("Subscription:", acct.get("name"))

print("Setting Azure subscription...")
az(["account", "set", "--subscription", SUBSCRIPTION_ID], expect_json=False)

print(f"Ensuring resource group exists: {RESOURCE_GROUP}")
rg = az([
    "group", "create",
    "--name", RESOURCE_GROUP,
    "--location", LOCATION
])

print("Resource group ready:", rg["name"])

In [ ]:
# ========= ENSURE LANGUAGE RESOURCE =========
def ensure_language_account(name, sku, location, resource_group):
    kind = "TextAnalytics"

    # 1) Check active resource
    existing = az_try([
        "cognitiveservices", "account", "show",
        "--name", name,
        "--resource-group", resource_group
    ])

    if existing is not None:
        print(f"Already exists: {name} ({kind})")
        return existing

    # 2) Check soft-deleted resource
    deleted = az_try([
        "cognitiveservices", "account", "show-deleted",
        "--name", name,
        "--resource-group", resource_group,
        "--location", location
    ])

    if deleted is not None:
        print(f"Recovering soft-deleted resource: {name} ({kind})")
        az([
            "cognitiveservices", "account", "recover",
            "--name", name,
            "--resource-group", resource_group,
            "--location", location
        ], expect_json=False)

        # Wait briefly for recovery to settle
        for _ in range(12):
            recovered = az_try([
                "cognitiveservices", "account", "show",
                "--name", name,
                "--resource-group", resource_group
            ])
            if recovered is not None:
                print(f"Recovered: {name} ({kind})")
                return recovered
            time.sleep(5)

        raise RuntimeError(f"Resource recovery started but '{name}' is not yet visible.")

    # 3) Create if it does not exist at all
    print(f"Creating: {name} ({kind})")
    created = az([
        "cognitiveservices", "account", "create",
        "--name", name,
        "--resource-group", resource_group,
        "--kind", kind,
        "--sku", sku,
        "--location", location
    ])

    return created


# ========= CREATE / ENSURE =========
language_account = ensure_language_account(
    LANGUAGE_RESOURCE_NAME,
    SKU,
    LOCATION,
    RESOURCE_GROUP
)

print("Azure AI Language resource ready.")

In [ ]:
# ========= FETCH ENDPOINT =========
language_endpoint = az([
    "cognitiveservices", "account", "show",
    "--name", LANGUAGE_RESOURCE_NAME,
    "--resource-group", RESOURCE_GROUP,
    "--query", "properties.endpoint"
], expect_json=False)

# ========= FETCH KEYS =========
language_keys = az([
    "cognitiveservices", "account", "keys", "list",
    "--name", LANGUAGE_RESOURCE_NAME,
    "--resource-group", RESOURCE_GROUP
])

language_key = language_keys.get("key1")

# ========= SAVE TO .env FILE =========
env_path = Path(".env")

env_values = {
    "AZURE_LANGUAGE_ENDPOINT": language_endpoint,
    "AZURE_LANGUAGE_KEY": language_key,
    "AZURE_LANGUAGE_RESOURCE_NAME": LANGUAGE_RESOURCE_NAME,
    "AZURE_RESOURCE_GROUP": RESOURCE_GROUP,
    "AZURE_LOCATION": LOCATION
}

# overwrite/create .env cleanly
with env_path.open("w", encoding="utf-8") as f:
    for k, v in env_values.items():
        f.write(f'{k}="{v}"\n')

print(f".env file written to: {env_path.resolve()}")
print("Language_Endpoint =", language_endpoint)
print("Language_Key      =", language_key [:6] + "...")